# Décodage d'un modèle de langage

Un grand modèle de langage (LLM) ne « répond » pas directement : pour un texte donné, il calcule une probabilité pour chaque *token* (morceau de mot) qui pourrait suivre. La manière de choisir le token suivant, le **décodage**, change beaucoup le résultat.

Nous utilisons ici un petit modèle ouvert, [Qwen2.5](https://huggingface.co/Qwen/Qwen2.5-1.5B) (licence Apache 2.0). Un GPU accélère la démonstration (`Exécution > Modifier le type d'exécution`), mais elle fonctionne aussi sur CPU.

In [ ]:
!pip install -q "transformers>=4.56,<5" accelerate

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

device = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL = "Qwen/Qwen2.5-1.5B"
CHAT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype="auto").to(device)
print(device)

## Des tokens, pas des mots

Le modèle ne voit pas des mots mais des *tokens* : des morceaux de mots issus d'un vocabulaire fixe.

In [ ]:
SENTENCE = "L'intelligence artificielle transforme les entreprises."

print(tokenizer.tokenize(SENTENCE))

## La probabilité du token suivant

Pour un début de phrase, le modèle donne une probabilité à chacun des ~150 000 tokens de son vocabulaire. Voici les 10 plus probables :

In [ ]:
CAPITAL_PROMPT = "La tour Eiffel se trouve à"

def next_token_table(prompt: str, k: int = 10) -> pd.DataFrame:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1]
    probabilities = torch.softmax(logits.float(), dim=-1)
    top = torch.topk(probabilities, k)
    return pd.DataFrame({
        "token": [repr(tokenizer.decode(i)) for i in top.indices],
        "probability": top.values.cpu().numpy().round(3),
    })


next_token_table(CAPITAL_PROMPT)

Le modèle a appris ces probabilités en lisant d'énormes quantités de texte : il n'a pas de base de faits, seulement des régularités statistiques.

## Décodage glouton

La stratégie la plus simple : prendre à chaque étape le token le plus probable, l'ajouter au texte, et recommencer.

In [ ]:
STORY_PROMPT = "Le meilleur moyen d'apprendre une langue étrangère, c'est"

def greedy(prompt: str, n_tokens: int = 40) -> str:
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    for _ in range(n_tokens):
        with torch.no_grad():
            logits = model(ids).logits[0, -1]
        next_id = logits.argmax()
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
    return tokenizer.decode(ids[0], skip_special_tokens=True)


print(greedy(STORY_PROMPT))

Le décodage glouton est déterministe : le même début donne toujours la même suite, souvent banale, et il tourne parfois en boucle.

## Échantillonnage et température

Plutôt que de prendre le plus probable, on peut **tirer au sort** le token suivant selon les probabilités du modèle. La **température** règle l'audace : basse, on reste sur les tokens les plus probables ; haute, on s'autorise des choix improbables.

In [ ]:
def generate(prompt: str, **kwargs) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=40, pad_token_id=tokenizer.eos_token_id,
                                **kwargs)
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
for temperature in [0.2, 0.8, 1.5]:
    set_seed(0)
    print(f"--- temperature = {temperature}")
    print(generate(STORY_PROMPT, do_sample=True, temperature=temperature, top_k=0))

À basse température le texte est sage ; à haute température il devient créatif, puis incohérent.

## Top-p : couper les choix improbables

Le *top-p* (ou *nucleus sampling*) ne tire au sort que parmi les tokens les plus probables dont la probabilité cumulée atteint `p` : on garde la variété sans les dérapages. C'est le réglage par défaut de la plupart des assistants. Trois tirages, trois textes différents :

In [ ]:
for seed in range(3):
    set_seed(seed)
    print(f"--- seed = {seed}")
    print(generate(STORY_PROMPT, do_sample=True, temperature=0.8, top_p=0.9))

## Modèle de base ou modèle « assistant » ?

Un modèle de base a seulement appris à **continuer** du texte. Posons-lui une question :

In [ ]:
QUESTION = "Quels sont trois avantages du télétravail pour une entreprise ?"

set_seed(0)
print(generate(QUESTION, do_sample=True, temperature=0.7, top_p=0.9))

Il continue le texte : parfois il répond, parfois il enchaîne sur d'autres questions ou dérive, car rien ne le pousse à répondre. Les assistants (ChatGPT, Claude, Le Chat, …) sont des modèles de base **affinés** pour suivre des instructions, puis alignés sur les préférences humaines. Même question à la version *Instruct* du même modèle :

In [ ]:
chat_tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL)
chat_model = AutoModelForCausalLM.from_pretrained(CHAT_MODEL, dtype="auto").to(device)

messages = [{"role": "user", "content": QUESTION}]
inputs = chat_tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                            return_dict=True, return_tensors="pt").to(device)
set_seed(0)
with torch.no_grad():
    output = chat_model.generate(**inputs, max_new_tokens=200, do_sample=True,
                                 temperature=0.7, top_p=0.9)
prompt_length = inputs["input_ids"].shape[1]
print(chat_tokenizer.decode(output[0, prompt_length:], skip_special_tokens=True))

## À retenir

- Un LLM calcule des probabilités de tokens suivants ; le texte dépend de la manière de choisir parmi eux
- Température et top-p règlent le compromis entre fiabilité et créativité
- Le même prompt peut donner des réponses différentes : il faut en tenir compte pour tester et évaluer
- Un assistant est un modèle de base affiné pour suivre des instructions